# Download das bibliotecas

In [14]:
!pip install pyswarm

# Importacao dos dados

In [15]:
import pandas as pd
import os
import numpy as np

In [16]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
caminho_dados = '/content/drive/Shareddrives/IC - Otimizacao computacional para inflacao/Dados'

# Listar os arquivos dentro da pasta "Dados"
arquivos_dados = os.listdir(caminho_dados)

print(arquivos_dados)


['dados_macro.ipynb', 'df_macro.csv']


In [18]:
# Caminho para o arquivo CSV
caminho_arquivo = os.path.join(caminho_dados, 'df_macro.csv')

# Ler o arquivo CSV
df = pd.read_csv(caminho_arquivo)

In [19]:
df = df.set_index('Date')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 154 entries, 2012-03-01 to 2024-12-01
Data columns (total 28 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   IPCA                                   154 non-null    float64
 1   IPCA_Transportes                       154 non-null    float64
 2   IPCA Alimentação_bebidas               154 non-null    float64
 3   INPC_Habitação                         154 non-null    float64
 4   IPCA_Saúde_cuidados_pessoais           154 non-null    float64
 5   IPCA_Vestuário                         154 non-null    float64
 6   IPCA_Educação                          154 non-null    float64
 7   IPCA_Despesas_Pessoais                 154 non-null    float64
 8   USDBRL                                 154 non-null    float64
 9   Reservas Internacionais                154 non-null    int64  
 10  Desemprego                             154 non-null    float64


# Importacao das ferramentas para implementacao do PSO

In [20]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from pyswarm import pso  # Biblioteca para PSO

# Funcao fitness

In [21]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# 📌 Função de fitness (não foi alterada)
def feature_selection_fitness(mask, X_train_scaled, y_train, X_test_scaled, y_test, model, metric_func):
    """
    Função de avaliação para PSO com modelo passado como argumento e métrica.
    """
    mask = np.round(mask).astype(int)  # Converter para 0 ou 1
    selected_features = X_train_scaled.columns[mask == 1]  # Selecionar as features

    if len(selected_features) == 0:  # Caso não tenha selecionado nenhuma feature
        return 9999  # Penalização por não selecionar nenhuma feature

    # Treinar o modelo com as features selecionadas
    model.fit(X_train_scaled[selected_features], y_train)

    # Fazer previsões
    y_pred = model.predict(X_test_scaled[selected_features])

    # Calcular a métrica
    score = metric_func(y_test, y_pred)

    return score  # Retorna a métrica calculada (MAE, MSE, etc.)

# Divisao dos dados

In [22]:

# 📌 Preparação dos dados
X = df.drop(columns=['IPCA'])  # Removendo IPCA das features
y = df['IPCA']

# Divisão entre treino e teste (SÉRIE TEMPORAL, sem shuffle)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

Normalizacao dos dados(media 0 e desvio padrao 1)

In [23]:
# 📌 Normalização dos dados
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)


# Implementacao usando random forest na funcao fitness

In [24]:
random_state = 42

In [25]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error  # <-- mudou aqui
from pyswarm import pso
import numpy as np

# 📌 Função de fitness corrigida para MAE (minimizar direto)
def constrained_feature_selection_fitness(mask, X_train, y_train, X_test, y_test, model, metric):
    mask = np.round(mask).astype(int)

    num_selected = mask.sum()
    if num_selected > 5 or num_selected == 0:
        # Penaliza fortemente
        return 1e6

    X_train_selected = X_train[:, mask == 1]
    X_test_selected = X_test[:, mask == 1]

    model.fit(X_train_selected, y_train)
    y_pred = model.predict(X_test_selected)

    # Aqui retorna o erro diretamente (minimizar MAE)
    return metric(y_test, y_pred)

# 📌 Configuração do PSO
num_features = X_train_scaled.shape[1]
lb = [0] * num_features
ub = [1] * num_features

# 📌 Rodar o PSO com MAE
best_mask, _ = pso(
    lambda mask: constrained_feature_selection_fitness(mask, X_train_scaled, y_train, X_test_scaled, y_test,
                                                       RandomForestRegressor(n_estimators=100, random_state=random_state),
                                                       mean_absolute_error),  # mudou aqui
    lb, ub,
    swarmsize=10,
    maxiter=20,
    debug=True
)

# 📌 Processar resultado
best_mask = np.round(best_mask).astype(int)

# ✅ Corrigir se selecionou mais de 5 features
if best_mask.sum() > 5:
    indices = np.argsort(-best_mask)[:5]
    new_mask = np.zeros_like(best_mask)
    new_mask[indices] = 1
    best_mask = new_mask

selected_features = X_train_scaled.columns[best_mask == 1]

print("🔹 Melhor conjunto de features selecionado pelo PSO (no máximo 5):", list(selected_features))


No constraints given.
Best after iteration 1: [0.26281427 0.58289405 0.17226071 0.75619384 0.10037126 0.13176568
 0.42664897 0.79812881 0.12893976 0.25233077 0.99939935 0.91903666
 0.95146441 0.23762466 0.28152143 0.94759655 0.76409717 0.12251803
 0.78112549 0.51207088 0.79791959 0.48334147 0.44044257 0.54793661
 0.2508367  0.76372532 0.87888792] 1000000.0
Best after iteration 2: [0.26281427 0.58289405 0.17226071 0.75619384 0.10037126 0.13176568
 0.42664897 0.79812881 0.12893976 0.25233077 0.99939935 0.91903666
 0.95146441 0.23762466 0.28152143 0.94759655 0.76409717 0.12251803
 0.78112549 0.51207088 0.79791959 0.48334147 0.44044257 0.54793661
 0.2508367  0.76372532 0.87888792] 1000000.0
Best after iteration 3: [0.26281427 0.58289405 0.17226071 0.75619384 0.10037126 0.13176568
 0.42664897 0.79812881 0.12893976 0.25233077 0.99939935 0.91903666
 0.95146441 0.23762466 0.28152143 0.94759655 0.76409717 0.12251803
 0.78112549 0.51207088 0.79791959 0.48334147 0.44044257 0.54793661
 0.2508367  